# Zadaća 14a

U zadaći 08a ste trenirali model logističke regresije na zadanom datasetu. Istrenirajte ga opet sa tim predanim kodom i provjerite vašnost značajki koristeći SHAP metodu. Provjerite važnost na cijelom modelu i na prvom retu testnih podataka.

In [42]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, ShuffleSplit
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
import shap

In [43]:
df = pd.read_csv("UCI_Credit_Card.csv", low_memory=False)
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


In [44]:
df.describe()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.00000,30000.000000,30000.000000,30000.000000,30000.000000
mean,15000.500000,167484.322667,1.603733,1.853133,1.551867,35.485500,-0.016700,-0.133767,-0.166200,-0.220667,...,43262.948967,40311.400967,38871.760400,5663.580500,5.921163e+03,5225.68150,4826.076867,4799.387633,5215.502567,0.221200
std,8660.398374,129747.661567,0.489129,0.790349,0.521970,9.217904,1.123802,1.197186,1.196868,1.169139,...,64332.856134,60797.155770,59554.107537,16563.280354,2.304087e+04,17606.96147,15666.159744,15278.305679,17777.465775,0.415062
min,1.000000,10000.000000,1.000000,0.000000,0.000000,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,...,-170000.000000,-81334.000000,-339603.000000,0.000000,0.000000e+00,0.00000,0.000000,0.000000,0.000000,0.000000
25%,7500.750000,50000.000000,1.000000,1.000000,1.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,2326.750000,1763.000000,1256.000000,1000.000000,8.330000e+02,390.00000,296.000000,252.500000,117.750000,0.000000
50%,15000.500000,140000.000000,2.000000,2.000000,2.000000,34.000000,0.000000,0.000000,0.000000,0.000000,...,19052.000000,18104.500000,17071.000000,2100.000000,2.009000e+03,1800.00000,1500.000000,1500.000000,1500.000000,0.000000
75%,22500.250000,240000.000000,2.000000,2.000000,2.000000,41.000000,0.000000,0.000000,0.000000,0.000000,...,54506.000000,50190.500000,49198.250000,5006.000000,5.000000e+03,4505.00000,4013.250000,4031.500000,4000.000000,0.000000
max,30000.000000,1000000.000000,2.000000,6.000000,3.000000,79.000000,8.000000,8.000000,8.000000,8.000000,...,891586.000000,927171.000000,961664.000000,873552.000000,1.684259e+06,896040.00000,621000.000000,426529.000000,528666.000000,1.000000


In [45]:
df.isna().sum()

ID                            0
LIMIT_BAL                     0
SEX                           0
EDUCATION                     0
MARRIAGE                      0
AGE                           0
PAY_0                         0
PAY_2                         0
PAY_3                         0
PAY_4                         0
PAY_5                         0
PAY_6                         0
BILL_AMT1                     0
BILL_AMT2                     0
BILL_AMT3                     0
BILL_AMT4                     0
BILL_AMT5                     0
BILL_AMT6                     0
PAY_AMT1                      0
PAY_AMT2                      0
PAY_AMT3                      0
PAY_AMT4                      0
PAY_AMT5                      0
PAY_AMT6                      0
default.payment.next.month    0
dtype: int64

In [46]:
target = df["default.payment.next.month"]
features_num = df.select_dtypes(include=[np.number])

imputer = SimpleImputer(strategy="median")
features_num_imputed = pd.DataFrame(imputer.fit_transform(features_num), columns=features_num.columns)

df_clean = pd.concat([features_num_imputed, target], axis=1)
features = df_clean.drop(columns=["default.payment.next.month"]).select_dtypes(include=[np.number])

In [47]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42, stratify=target)

In [48]:
log_reg = LogisticRegression(solver='saga', max_iter=50000)
log_reg.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'saga'
,max_iter,50000
,multi_class,'deprecated'


In [49]:
explainer = shap.LinearExplainer(model=log_reg, masker=X_train)
shap_values = explainer.shap_values(X_test)

In [50]:
print("Prosječna apsolutna SHAP važnost po značajci:")
vaznost = np.abs(shap_values).mean(axis=0)
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Mean |SHAP value|': vaznost
}).sort_values(by='Mean |SHAP value|', ascending=False)
print(feature_importance.head(10))

Prosječna apsolutna SHAP važnost po značajci:
      Feature  Mean |SHAP value|
1   LIMIT_BAL           0.450843
12  BILL_AMT1           0.401023
18   PAY_AMT1           0.214751
0          ID           0.190667
13  BILL_AMT2           0.181298
19   PAY_AMT2           0.151193
15  BILL_AMT4           0.147676
16  BILL_AMT5           0.078347
14  BILL_AMT3           0.063322
17  BILL_AMT6           0.062228


In [51]:
print("\nObjašnjenje za prvi red testnog skupa:")
first_row = X_test.iloc[0:1]
shap_value_first = explainer.shap_values(first_row)

shap_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Feature value': first_row.values[0],
    'SHAP value': shap_value_first[0]
}).sort_values(by='SHAP value', ascending=False)

print(shap_df.head(10))


Objašnjenje za prvi red testnog skupa:
      Feature  Feature value  SHAP value
1   LIMIT_BAL        50000.0    0.521722
12  BILL_AMT1         1540.0    0.376738
0          ID         6908.0    0.177092
19   PAY_AMT2            0.0    0.142287
22   PAY_AMT5         1764.0    0.019349
23   PAY_AMT6         2841.0    0.006223
21   PAY_AMT4         2320.0    0.004953
2         SEX            1.0    0.000002
10      PAY_5            0.0    0.000002
11      PAY_6            0.0    0.000002
